In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from torchvision import transforms
import torchvision

import matplotlib.pyplot as plt

from collections import namedtuple

from sklearn.metrics import classification_report

In [2]:
def get_clases():
  classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
  return classes

TrainTest = namedtuple('TrainTest', ['train', 'test'])

def prepare_data():
  # tham khảo thêm https://github.com/albumentations-team/albumentations
  transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
  ])
  transform_test = transforms.Compose([
    transforms.ToTensor()
  ])
  trainset = torchvision.datasets.CIFAR10(root='./data', download=True, train=True, transform=transform_train)
  testset = torchvision.datasets.CIFAR10(root='./data', download=True, train=False, transform=transform_test)
  return TrainTest(train=trainset, test=testset)

def prepare_loader(datasets):
  trainloader = DataLoader(dataset=datasets.train, batch_size=128, shuffle=True, num_workers=4)
  testloader = DataLoader(dataset=datasets.test, batch_size=128, shuffle=False, num_workers=4)
  return TrainTest(train=trainloader, test=testloader)

class VGG16(nn.Module):
  def __init__(self):
    super().__init__()
    self.features = self._make_features()
    self.classification_head = nn.Linear(in_features=512, out_features=10)

  def forward(self, x):
    out = self.features(x)
    out = out.view(out.size(0), -1)
    out = self.classification_head(out)
    return out

  def _make_features(self):
    config = [64,64,'MP',128,128,'MP',256,256,256,'MP',512,512,512,'MP',512,512,512,'MP']
    layers = []
    c_in = 3
    for c in config:
      if c == 'MP':
        layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
      else:
        layers += [nn.Conv2d(in_channels=c_in, out_channels=c, kernel_size=3, stride=1, padding=1),
                   nn.BatchNorm2d(num_features=c),
                   nn.ReLU6(inplace=True)]
        c_in = c
    return nn.Sequential(*layers)

def imshow(images, labels, predicted, target_names):
  img = torchvision.utils.make_grid(images)
  plt.imshow(img.permute(1,2,0).cpu().numpy())
  [print(target_names[c], end=' ') for c in list(labels.cpu().numpy()) ]
  print()
  [print(target_names[c], end=' ') for c in list(predicted.cpu().numpy()) ]
  print()

def train_epoch(epoch, model, loader, loss_func, optimizer, device):
  model.train()
  running_loss = 0.0
  reporting_steps = 60
  for i, (images, labels) in enumerate(loader):
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    loss = loss_func(outputs, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
    if i % reporting_steps == reporting_steps-1:
      print(f"Epoch {epoch} step {i} ave_loss {running_loss/reporting_steps:.4f}")
      running_loss = 0.0

def test_epoch(epoch, model, loader, device):
  ytrue = []
  ypred = []
  with torch.no_grad():
    model.eval()

    for i, (images, labels) in enumerate(loader):
      images, labels = images.to(device), labels.to(device)
      outputs = model(images)
      _, predicted = torch.max(outputs, dim=1)

      ytrue += list(labels.cpu().numpy())
      ypred += list(predicted.cpu().numpy())

  return ypred, ytrue

def main(PATH='./model.pth'):
  classes = get_clases()
  datasets = prepare_data()
  # img, label = datasets.train[0]
  # plt.imshow(img)
  # print(classes[label], img.size)
  # print('train', len(datasets.train), 'test', len(datasets.test))

  loaders = prepare_loader(datasets)
  # images, labels = iter(loaders.train).next()
  # print(images.shape, labels.shape)

  device = torch.device("cuda:0")
  model = VGG16().to(device)
  # images, labels = iter(loaders.train).next()
  # outputs = model(images)
  # print(outputs.shape)
  # print(outputs[0])
  # _, predicted = torch.max(outputs, dim=1)
  # print(predicted)
  # imshow(images, labels, predicted, classes)

  loss_func = nn.CrossEntropyLoss()
  optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
  for epoch in range(10):
    train_epoch(epoch, model, loaders.train, loss_func, optimizer, device)
    ypred, ytrue = test_epoch(epoch, model, loaders.test, device)
    print(classification_report(ytrue, ypred, target_names=classes))

    torch.save(model.state_dict(), PATH)

  return model

model = main()

100%|██████████| 170M/170M [00:10<00:00, 16.1MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_ratio

Epoch 0 step 59 ave_loss 2.1394
Epoch 0 step 119 ave_loss 1.7554
Epoch 0 step 179 ave_loss 1.5109
Epoch 0 step 239 ave_loss 1.4349
Epoch 0 step 299 ave_loss 1.2999
Epoch 0 step 359 ave_loss 1.2053
              precision    recall  f1-score   support

       plane       0.38      0.31      0.34      1000
         car       0.47      0.82      0.60      1000
        bird       0.36      0.35      0.36      1000
         cat       0.29      0.46      0.36      1000
        deer       0.51      0.07      0.12      1000
         dog       1.00      0.00      0.01      1000
        frog       0.48      0.82      0.60      1000
       horse       0.92      0.24      0.38      1000
        ship       0.36      0.95      0.52      1000
       truck       0.87      0.14      0.24      1000

    accuracy                           0.42     10000
   macro avg       0.56      0.42      0.35     10000
weighted avg       0.56      0.42      0.35     10000



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 1 step 59 ave_loss 1.1120
Epoch 1 step 119 ave_loss 1.0325
Epoch 1 step 179 ave_loss 0.9718
Epoch 1 step 239 ave_loss 0.9551
Epoch 1 step 299 ave_loss 0.9080
Epoch 1 step 359 ave_loss 0.8997
              precision    recall  f1-score   support

       plane       0.69      0.65      0.67      1000
         car       0.90      0.81      0.85      1000
        bird       0.59      0.53      0.56      1000
         cat       0.49      0.17      0.25      1000
        deer       0.69      0.62      0.65      1000
         dog       0.42      0.77      0.54      1000
        frog       0.93      0.46      0.62      1000
       horse       0.84      0.57      0.68      1000
        ship       0.52      0.94      0.67      1000
       truck       0.70      0.85      0.77      1000

    accuracy                           0.64     10000
   macro avg       0.68      0.64      0.63     10000
weighted avg       0.68      0.64      0.63     10000



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


...

Epoch 7 step 59 ave_loss 0.3967
Epoch 7 step 119 ave_loss 0.4322
Epoch 7 step 179 ave_loss 0.4392
Epoch 7 step 239 ave_loss 0.4308
Epoch 7 step 299 ave_loss 0.4171
Epoch 7 step 359 ave_loss 0.4226
              precision    recall  f1-score   support

       plane       0.67      0.93      0.78      1000
         car       0.74      0.98      0.85      1000
        bird       0.82      0.69      0.75      1000
         cat       0.68      0.68      0.68      1000
        deer       0.89      0.65      0.75      1000
         dog       0.75      0.77      0.76      1000
        frog       0.78      0.91      0.84      1000
       horse       0.96      0.72      0.82      1000
        ship       0.95      0.83      0.88      1000
       truck       0.89      0.80      0.84      1000

    accuracy                           0.80     10000
   macro avg       0.81      0.80      0.79     10000
weighted avg       0.81      0.80      0.79     10000



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 8 step 59 ave_loss 0.3858
Epoch 8 step 119 ave_loss 0.3940
Epoch 8 step 179 ave_loss 0.3864
Epoch 8 step 239 ave_loss 0.3839
Epoch 8 step 299 ave_loss 0.4046
Epoch 8 step 359 ave_loss 0.3770
              precision    recall  f1-score   support

       plane       0.73      0.94      0.82      1000
         car       0.92      0.94      0.93      1000
        bird       0.87      0.70      0.77      1000
         cat       0.69      0.71      0.70      1000
        deer       0.83      0.80      0.81      1000
         dog       0.75      0.77      0.76      1000
        frog       0.95      0.78      0.86      1000
       horse       0.83      0.89      0.86      1000
        ship       0.86      0.92      0.89      1000
       truck       0.96      0.84      0.90      1000

    accuracy                           0.83     10000
   macro avg       0.84      0.83      0.83     10000
weighted avg       0.84      0.83      0.83     10000



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 9 step 59 ave_loss 0.3490
Epoch 9 step 119 ave_loss 0.3633
Epoch 9 step 179 ave_loss 0.3732
Epoch 9 step 239 ave_loss 0.3627
Epoch 9 step 299 ave_loss 0.3491
Epoch 9 step 359 ave_loss 0.3568
              precision    recall  f1-score   support

       plane       0.82      0.90      0.86      1000
         car       0.94      0.91      0.92      1000
        bird       0.60      0.91      0.72      1000
         cat       0.77      0.63      0.70      1000
        deer       0.87      0.81      0.84      1000
         dog       0.89      0.65      0.75      1000
        frog       0.83      0.93      0.88      1000
       horse       0.95      0.84      0.89      1000
        ship       0.89      0.91      0.90      1000
       truck       0.94      0.88      0.91      1000

    accuracy                           0.84     10000
   macro avg       0.85      0.84      0.84     10000
weighted avg       0.85      0.84      0.84     10000

